<!-- Media Lite: open widget cell ①, then run cell ②. Help and Undo live in the UI. -->

In [ ]:
#@title 🎛️ ① افتح Media Lite Smart Tabs { display-mode: "form" }
import importlib.util
import json
import os
from pathlib import Path
import re
import subprocess
import sys
import urllib.request

_UI_REPO = "abdullahsamirashour/gpt"
_UI_REF = "main"  # use the same stable branch as the existing Colab link
_UI_FILE = "telegram-smart-compressor/smart_tabs_ui.py"
_UI_LOCAL = Path("/content/media_lite_smart_tabs_ui.py")
_UI_MARKER = Path("/content/media_lite_smart_tabs_ui.py.commit")

def _ui_ref_sha():
    proc = subprocess.run(
        ["git", "ls-remote", f"https://github.com/{_UI_REPO}.git",
         f"refs/heads/{_UI_REF}"],
        capture_output=True, text=True, timeout=20,
    )
    candidate = proc.stdout.strip().split()[0] if proc.returncode == 0 and proc.stdout.strip() else ""
    if re.fullmatch(r"[0-9a-f]{40}", candidate):
        return candidate
    req = urllib.request.Request(
        f"https://api.github.com/repos/{_UI_REPO}/git/ref/heads/{_UI_REF}",
        headers={"User-Agent": "Media-Lite-Tabs", "Cache-Control": "no-cache"},
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        sha = json.load(response)["object"]["sha"]
    if not re.fullmatch(r"[0-9a-f]{40}", sha):
        raise RuntimeError("GitHub commit غير صالح.")
    return sha

try:
    _ui_sha = _ui_ref_sha()
    if (not _UI_LOCAL.is_file() or not _UI_MARKER.is_file()
            or _UI_MARKER.read_text(encoding="ascii").strip() != _ui_sha):
        req = urllib.request.Request(
            f"https://raw.githubusercontent.com/{_UI_REPO}/{_ui_sha}/{_UI_FILE}",
            headers={"User-Agent": "Media-Lite-Tabs"},
        )
        with urllib.request.urlopen(req, timeout=45) as response:
            _ui_src = response.read().decode("utf-8")
        if len(_ui_src) < 10000:
            raise RuntimeError("Smart Tabs module لم يتحمّل كاملًا.")
        temp = _UI_LOCAL.with_suffix(".tmp")
        temp.write_text(_ui_src, encoding="utf-8")
        os.replace(temp, _UI_LOCAL)
        _UI_MARKER.write_text(_ui_sha, encoding="ascii")

    _ui_spec = importlib.util.spec_from_file_location("media_lite_smart_tabs", _UI_LOCAL)
    _ui_module = importlib.util.module_from_spec(_ui_spec)
    sys.modules[_ui_spec.name] = _ui_module
    _ui_spec.loader.exec_module(_ui_module)
    MEDIA_LITE_UI = _ui_module.launch()
    print("✅ الواجهة جاهزة. شغّل ② للتنفيذ.")

except Exception as _ui_error:
    # A backend or front-end widget problem must not strand a mobile user.
    # Keep the original Colab Forms available independently of these widgets.
    MEDIA_LITE_UI = None
    from IPython.display import Markdown, display
    print("⚠️ تعذّر فتح Smart UI:", type(_ui_error).__name__, str(_ui_error))
    display(Markdown(
        "📱 **مشكلة في الواجهة؟** "
        "[افتح النسخة الكلاسيكية](https://colab.research.google.com/"
        "github/abdullahsamirashour/gpt/blob/"
        "backup/media-lite-pre-tabs-2026-11-20/"
        "telegram-smart-compressor/Smart_Compressor.ipynb) "
        "بنفس أدوات الضغط والنقل."
    ))


In [ ]:
#@title ▶ ② نفّذ الأداة المختارة (Stop من زر Colab الأصلي) { display-mode: "form" }
if not globals().get("MEDIA_LITE_UI"):
    raise RuntimeError("شغّل خلية ① أولًا لفتح الـSmart Tabs.")
MEDIA_LITE_UI.execute_selected()


> 📱 الواجهة مش شغالة عندك؟ [افتح النسخة الكلاسيكية بـColab Forms](https://colab.research.google.com/github/abdullahsamirashour/gpt/blob/backup/media-lite-pre-tabs-2026-11-20/telegram-smart-compressor/Smart_Compressor.ipynb).